# Step 2: Set Up Your Geospatial Notebook

This notebook loads spatial data (field boundaries) and soil data, verifies CRS alignment, and prepares data for geospatial analysis.

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully")

## 1. Load Field Boundaries (GeoJSON)

In [ ]:
# Load field boundaries from GeoJSON
fields = gpd.read_file('../data/michigan_fields.geojson')

print(f"Loaded {len(fields)} fields")
print(f"Columns: {list(fields.columns)}")
print(f"\nCRS: {fields.crs}")
fields.head()

## 2. Load Soil Data (SSURGO)

In [ ]:
# Load soil data from SSURGO (downloaded via NRCS API)
soil = pd.read_csv('../data/michigan_soil_ssurgo.csv')

print(f"Loaded {len(soil)} soil records")
print(f"Fields with soil data: {soil['field_id'].nunique()}")
print(f"\nColumns: {list(soil.columns)}")
soil.head()

## 3. Check CRS - Verify Alignment

In [ ]:
# Check CRS of field boundaries
print("=== CRS CHECK ===")
print(f"Field boundaries CRS: {fields.crs}")
print(f"Field boundaries CRS (EPSG): {fields.crs.to_epsg()}")

# Soil data is attribute data (CSV) - no spatial CRS
# When joined to fields, the combined layer inherits field CRS
print(f"\nSoil data: Attribute table (CSV) - no spatial CRS")
print("Note: Soil data will be joined to field boundaries using 'field_id'")
print("      The combined layer will inherit field boundary CRS (EPSG:4326)")

## 4. Join Soil Data to Field Boundaries

In [ ]:
# Get dominant soil per field (first row per field_id, sorted by comppct_r)
dominant_soil = soil.sort_values('comppct_r', ascending=False).groupby('field_id').first().reset_index()
dominant_soil = dominant_soil[['field_id', 'mukey', 'muname', 'compname', 'comppct_r', 
                                'drainagecl', 'om_r', 'ph1to1h2o_r', 'claytotal_r', 'sandtotal_r']]

# Join to fields
fields_with_soil = fields.merge(dominant_soil, on='field_id', how='left')

print(f"Fields with soil joined: {len(fields_with_soil)}")
print(f"CRS after join: {fields_with_soil.crs}")
fields_with_soil[['field_id', 'county', 'crop_name', 'compname', 'om_r', 'ph1to1h2o_r']].head()

## 5. Summary - CRS Verification

In [ ]:
# Final CRS verification
print("=== FINAL CRS VERIFICATION ===")
print(f"Combined layer CRS: {fields_with_soil.crs}")
print(f"CRS is WGS84 (EPSG:4326): {fields_with_soil.crs.to_epsg() == 4326}")

# Check CRS type
print(f"\nCRS type: {fields_with_soil.crs.geodetic_crs.name if hasattr(fields_with_soil.crs, 'geodetic_crs') else 'Geographic'}")
print("\n✅ Both layers use the same CRS (EPSG:4326)")
print("   - Field boundaries: EPSG:4326 (WGS84)")
print("   - Soil data: Joined as attributes, inherits field CRS")

## 6. Quick Visualization

In [ ]:
# Simple visualization of fields with soil data
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

fields_with_soil.plot(column='om_r', ax=ax, legend=True, 
                       legend_kwds={'label': 'Organic Matter (%)'},
                       cmap='Greens', edgecolor='black')

ax.set_title('Field Boundaries - Organic Matter Content')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()